### Funciones de Orden Superior
> 1. Las funciones de orden superior son funciones que operan sobre tipos de datos complejos, como: JSON, Arrays y Maps.
> 2. Permiten pasar funciones como argumentos (como expresiones lambda), aplicar transformaciones y devolver Arrays o Maps.
> 3. Son extremadamente útiles para manipular Arrays sin hacerlas explotar.

#### Uso común de funciones de Arrays de orden superior
- TRANSFORM
- FILTER
- EXISTS
- AGGREGATE

#### Sintaxis:
<hr>

`<function_name> (array_column, lambda expression)` <br>
lambda_expression: `element -> expression`

In [0]:
CREATE OR REPLACE TEMPORARY VIEW product_items
AS
  SELECT * FROM
  VALUES
    (1, array('keyboard', 'monitor', 'mouse', 'smartphone')),
    (2, array('tablet', 'ram memory', 'smartwatch')),
    (3, array('printer', 'laptop'))
  AS products(product_id, items)

In [0]:
SELECT * FROM product_items;

##### 1. Convierte todos los nombres de los elementos a MAYÚSCULAS (Función TRANSFORM)

In [0]:
SELECT product_id,
       TRANSFORM(items, x -> UPPER(x)) AS upper_items
FROM product_items;

##### 2. Filtrar solo los elementos que contengan la cadena 'smart' (Función FILTER)

In [0]:
SELECT product_id,
       FILTER(items, x -> x LIKE '%smart%') AS smart_items
FROM product_items;

##### 3. Comprobar si existe un producto 'tablet' (función EXISTS).

In [0]:
SELECT product_id,
       EXISTS(items, x -> x = 'tablet') AS has_items
FROM product_items;

#### Array con más de un objeto

In [0]:
CREATE OR REPLACE TEMPORARY VIEW product_items
AS
  SELECT * FROM
  VALUES
    (1, array(
            named_struct('name', 'keyboard', 'price', 99),
            named_struct('name', 'monitor', 'price', 899),
            named_struct('name', 'mouse', 'price', 169),
            named_struct('name', 'smartphone', 'price', 799)
        )),
    (2, array(
            named_struct('name', 'tablet', 'price', 359),
            named_struct('name', 'ram memory', 'price', 89),
            named_struct('name', 'smartwatch', 'price', 329)
        )),
    (3, array(
            named_struct('name', 'printer', 'price', 299),
            named_struct('name', 'laptop', 'price', 1299)
        ))
  AS products(product_id, items)

In [0]:
SELECT * FROM product_items;

##### 1. Convertit todos los nombres de los artículos a MAYÚSCULAS y agregue un 18% de IMPUESTO a cada artículo (Función TRANSFORM).

In [0]:
SELECT product_id,
       TRANSFORM(items, x -> named_struct(
                                          'name', UPPER(x.name),
                                          'price', ROUND(x.price * 1.18, 2)
                                         )) AS item_with_tax
FROM product_items;

##### 2. Calcular el "importe total" del producto para cada "product_id" (Función AGGREGATE)

In [0]:
SELECT product_id,
       AGGREGATE(items, 0, (acc, x) -> acc + x.price) AS total_product_price
FROM product_items;

#### Función Maps
<hr>

> Un mapa es una colección de pares clave-valor, como un diccionario. <br>`{'smartphone': 1100, 'tablet': 600}`

#### Uso común de funciones "maps" de orden superior
- TRANSFORM_VALUES
- TRANSFORM_KEYS
- MAP_FILTER

#### Sintaxis:
<hr>

`<function_name> (map_column, lambda_expression)` <br>
_lambda expression:_ `(key_value) -> expression`

In [0]:
CREATE OR REPLACE TEMPORARY VIEW product_item_prices
AS
  SELECT * FROM VALUES
    (1, map('keyboard', 99, 'monitor', 899, 'mouse', 169, 'smartphone', 799)),
    (2, map('tablet', 359, 'ram memory', 89, 'smartwatch', 329)),
    (3, map('printer', 299, 'laptop', 1299))
  AS products(product_id, item_prices)

In [0]:
SELECT * FROM product_item_prices;

##### 1. Convierte todos los nombres de los elementos a MAYÚSCULAS (Función TRANSFORM_KEYS)

In [0]:
SELECT product_id,
       TRANSFORM_KEYS(item_prices, (item, price) -> UPPER(item)) AS items_upper_case
FROM product_item_prices;

##### 2. Aplicar un 18% de IMPUESTO a los precios de los artículos (Función TRANSFORM_VALUES)

In [0]:
SELECT product_id,
       TRANSFORM_VALUES(item_prices, (item, price) -> ROUND(price * 1.18, 2)) AS items_with_tax
FROM product_item_prices;

##### 3. Filtrar solo los artículos con un precio superior a 700 dólares (Función MAP_FILTER)

In [0]:
SELECT product_id,
       MAP_FILTER(item_prices, (item, price) -> price > 700) AS primiun_items
FROM product_item_prices;